In [1]:
import numpy as np
import pandas as pd

# Definir as variáveis de caminho
DATADIR = 'C:/Users/Usuário/Downloads/machineLearning/dataset/'
TABLEDIR = DATADIR+'data_tests.csv'
IMAGEDIR = DATADIR+'images'

# Ler os valores tabulares
df = pd.read_csv(TABLEDIR)
# Verificar o número total de entradas
print("O número total de entradas é: ", df.shape[0], "\n")
# Verificar celulas vazias na tabela
print("O número de células vazias por coluna é: \n", df.isnull().sum())

O número total de entradas é:  33126 

O número de células vazias por coluna é: 
 image_name                         4
patient_id                         0
sex                               65
age_approx                        68
anatom_site_general_challenge    527
diagnosis                          0
benign_malignant                   0
target                             0
dtype: int64


In [2]:
# Como podemos ver, algumas das células não possuem valor. A coluna com a maior concentração de células vazias é anatom_site_general_challenge
# A quantidade de células vazias corresponde a 1.59% das entradas.
uniqueAnatom = df["anatom_site_general_challenge"].value_counts(dropna=False)
print(uniqueAnatom, "\n")

# Para não perdermos amostras, as células vazias categóricas (anatom e sex) serão preenchidas como 'unknown', enquanto as células vazias contínuas (age) serão preemchidas com a média
mean_age = df["age_approx"].mean()
df["age_approx"] = df["age_approx"].fillna(mean_age)
df = (df.fillna("unknown"))
print("O número de células vazias por coluna é: \n", df.isnull().sum())

anatom_site_general_challenge
torso              16845
lower extremity     8417
upper extremity     4983
head/neck           1855
NaN                  527
palms/soles          375
oral/genital         124
Name: count, dtype: int64 

O número de células vazias por coluna é: 
 image_name                       0
patient_id                       0
sex                              0
age_approx                       0
anatom_site_general_challenge    0
diagnosis                        0
benign_malignant                 0
target                           0
dtype: int64


In [3]:
# Podemos notar que algumas colunas possuem valores de texto, então iremos mapear 
# Originalmente usava o LabelEncoder mas depois de pesquisar mais, descobri que o OneHotEncoder é melhor nesse caso.
# As colunas 'diagnosis' e 'benign_malignant' não vão ser codificadas, pois elas podem causar vazamento de dados

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(df[['anatom_site_general_challenge']])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['anatom_site_general_challenge']), index=df.index)
df = pd.concat([df.drop(columns=['anatom_site_general_challenge']), encoded_df], axis=1)

encoded = encoder.fit_transform(df[['sex']])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['sex']), index=df.index)
df = pd.concat([df.drop(columns=['sex']), encoded_df], axis=1)

In [4]:
# Agora vamos iniciar o pre-processamento das imagens
# Primeiro vamos criar uma nova coluna "caminho_imagem" para definir onde a imagem atrelada aquela entrada se encontra
import os

df["caminho_imagem"] = df["image_name"].apply(lambda x: os.path.join(IMAGEDIR, f"{x}.jpg"))
df["caminho_imagem"] = df["caminho_imagem"].str.replace("\\", "/", regex=False)
print(df[["image_name", "caminho_imagem"]].head())


# Agora vamos verificar se todas as entradas possuem uma imagem relacionada
df["existe"] = df["caminho_imagem"].apply(os.path.exists)
print("\n", df["existe"].value_counts())
# Se uma entrada não possuir imagem, ela será removida
df = df[df["existe"]].copy()
print(df.shape)


     image_name                                     caminho_imagem
0       unknown  C:/Users/Usuário/Downloads/machineLearning/dat...
1       unknown  C:/Users/Usuário/Downloads/machineLearning/dat...
2  ISIC_0052212  C:/Users/Usuário/Downloads/machineLearning/dat...
3       unknown  C:/Users/Usuário/Downloads/machineLearning/dat...
4  ISIC_0074268  C:/Users/Usuário/Downloads/machineLearning/dat...

 existe
True     33122
False        4
Name: count, dtype: int64
(33122, 18)


In [5]:
import keras
image_input = keras.Input(shape=(128, 128, 3))

x = keras.layers.Conv2D(32, 3, activation="relu")(image_input)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Conv2D(64, 3, activation="relu")(x)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Flatten()(x)

image_features = keras.layers.Dense(128, activation="relu")(x)

In [6]:
tabular_input = keras.Input(shape=(11,))

y = keras.layers.Dense(64, activation="relu")(tabular_input)
y = keras.layers.Dense(32, activation="relu")(y)

tabular_features = keras.layers.Dense(16, activation="relu")(y)

In [7]:
combined = keras.layers.concatenate([image_features, tabular_features])

z = keras.layers.Dense(64, activation="relu")(combined)
z = keras.layers.Dropout(0.3)(z)

output = keras.layers.Dense(1, activation="sigmoid")(z)

In [8]:
model = keras.Model(
    inputs=[image_input, tabular_input],
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [15]:
import cv2
from PIL import Image
import numpy as np

X_images = []

for path in df["caminho_imagem"]:
    img = Image.open(path).convert("RGB")
    img = np.array(img)
    img = img / 255.0

    X_images.append(img)

X_images = np.array(X_images)

In [ ]:
X_tabular = df.drop(columns=["image_name","patient_id","caminho_imagem","existe","benign_malignant", "target", "diagnosis"])
y = df["target"]



In [28]:
# Agora, vamos separar os conjuntos de treino, validação e teste
from sklearn.model_selection import train_test_split

X_train_val_tab, X_test_tab, X_train_val_img, X_test_img, y_train_val, y_test = train_test_split(
    X_tabular,
    X_images,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train_tab, X_val_tab, X_train_img, X_val_img, y_train, y_val = train_test_split(
    X_train_val_tab,
    X_train_val_img,
    y_train_val,
    test_size=0.2,
    stratify=y_train_val,
    random_state=42
)

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_tab = scaler.fit_transform(X_train_tab)
X_val_tab = scaler.transform(X_val_tab)
X_test_tab = scaler.transform(X_test_tab)

In [30]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = dict(zip(classes, weights))

In [33]:
model.fit(
    [X_train_img, X_train_tab],
    y_train,
    epochs=5,
    batch_size=32,
    class_weight=class_weight,
     validation_data=([X_val_img, X_val_tab], y_val)
)

Epoch 1/5
663/663 ━━━━━━━━━━━━━━━━━━━━ 105s 159ms/step - accuracy: 0.7955 - loss: 0.6036 - val_accuracy: 0.9732 - val_loss: 0.4618
Epoch 2/5
663/663 ━━━━━━━━━━━━━━━━━━━━ 104s 157ms/step - accuracy: 0.8199 - loss: 0.5668 - val_accuracy: 0.8240 - val_loss: 0.5928
Epoch 3/5
663/663 ━━━━━━━━━━━━━━━━━━━━ 104s 156ms/step - accuracy: 0.7987 - loss: 0.5591 - val_accuracy: 0.7626 - val_loss: 0.6384
Epoch 4/5
663/663 ━━━━━━━━━━━━━━━━━━━━ 104s 157ms/step - accuracy: 0.8363 - loss: 0.5209 - val_accuracy: 0.8319 - val_loss: 0.5231
Epoch 5/5
663/663 ━━━━━━━━━━━━━━━━━━━━ 104s 156ms/step - accuracy: 0.8274 - loss: 0.5399 - val_accuracy: 0.6817 - val_loss: 0.6935


In [36]:
from sklearn.metrics import classification_report

y_pred = (model.predict([X_test_img,X_test_tab]) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step
              precision    recall  f1-score   support

           0       1.00      0.68      0.80      6508
           1       0.04      0.83      0.08       117

    accuracy                           0.68      6625
   macro avg       0.52      0.75      0.44      6625
weighted avg       0.98      0.68      0.79      6625



In [37]:
model.evaluate([X_test_img, X_test_tab], y_test)

208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.6780 - loss: 0.6882


[0.6881669163703918, 0.6780377626419067]

In [38]:
from sklearn.metrics import roc_auc_score

probs = model.predict([X_test_img,X_test_tab])

auc = roc_auc_score(y_test, probs)

print("ROC AUC =", auc)

208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step
ROC AUC = 0.8517314652840161
